In [3]:
import numpy as np
import pandas as pd

import statsmodels.api as sm

import seaborn as sns
import matplotlib.pylab as plt

from sklearn.metrics import r2_score as r2
from sklearn.metrics import mean_absolute_percentage_error as mape

from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

Есть данные о продажах, остатках, закупках по клиентам за период неполных два года
Хотим прогнозировать продажи с учетом сезонности, остатков и закупок
Метод выбрали Категориальный бустинг, чтобы наименование клиента передать как категориальный признак
Сезонность продаж зависит от месяца - поэтому он тоже пойдет как категория
Числовыми признаками будут -  номер периода, остатки, закупки

In [5]:
#! pip install catboost
from catboost import CatBoostRegressor

In [61]:
df = pd.read_excel('Case_Analyst_Решение.xlsx', sheet_name  = 'data').fillna(0)
df.head()

,Период,Клиент,Продажи,Остатки,Закупки
0,2020-01-01,Григорий,1.680468e+08,2.994128e+08,2.095030e+08
1,2020-02-01,Григорий,2.036499e+08,3.381993e+08,1.539659e+08
2,2020-03-01,Григорий,2.566368e+08,3.322793e+08,3.491804e+08
3,2020-04-01,Григорий,2.317790e+08,4.224878e+08,2.521181e+08
4,2020-05-01,Григорий,1.952722e+08,3.031453e+08,1.931633e+08


In [75]:
#отделяю актуальные данные на которых буду учить и валидировать модель
df_actual = df[(df['Период'] < '2021-10')].copy()
df_actual.tail()

,Период,Клиент,Продажи,Остатки,Закупки
136,2021-05-01,Другие,7.570116e+08,3.995233e+08,1.652844e+08
137,2021-06-01,Другие,8.973925e+08,3.999651e+08,1.918143e+08
138,2021-07-01,Другие,8.957503e+08,4.479332e+08,1.144805e+08
139,2021-08-01,Другие,8.427083e+08,4.091580e+08,1.630058e+08
140,2021-09-01,Другие,9.661186e+08,3.669062e+08,1.828301e+08


In [76]:
#меняю дату на номер периода чтобы использовать как фичу
df_actual['Номер_периода'] = df_actual.groupby('Клиент')['Период'].rank(method='max')

#месяц тоже нужен как влияющий на продажи фактор (сезонность)
df_actual['Месяц'] = df_actual['Период'].dt.month.astype('object')


df_actual.tail()

,Период,Клиент,Продажи,Остатки,Закупки,Номер_периода,Месяц
136,2021-05-01,Другие,7.570116e+08,3.995233e+08,1.652844e+08,17.0,5
137,2021-06-01,Другие,8.973925e+08,3.999651e+08,1.918143e+08,18.0,6
138,2021-07-01,Другие,8.957503e+08,4.479332e+08,1.144805e+08,19.0,7
139,2021-08-01,Другие,8.427083e+08,4.091580e+08,1.630058e+08,20.0,8
140,2021-09-01,Другие,9.661186e+08,3.669062e+08,1.828301e+08,21.0,9


In [77]:
df_actual.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 126 entries, 0 to 140
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Период         126 non-null    datetime64[ns]
 1   Клиент         126 non-null    object        
 2   Продажи        126 non-null    float64       
 3   Остатки        126 non-null    float64       
 4   Закупки        126 non-null    float64       
 5   Номер_периода  126 non-null    float64       
 6   Месяц          126 non-null    object        
dtypes: datetime64[ns](1), float64(4), object(2)
memory usage: 7.9+ KB


In [78]:
#делю на тест и трейн, затем тест еще на тест и валидацию для контроля переобучения модели в соотношении 60\20\20
train, test = train_test_split(df_actual,train_size=0.8,random_state=42)

val, test = train_test_split(test, train_size=0.5,random_state=42)


In [79]:
#список фичей
X = ['Номер_периода','Месяц','Клиент','Остатки','Закупки']

#из них категориальные
cat_features = ['Месяц','Клиент']

#целевая переменная - что хотим предсказывать
y = ['Продажи']

In [80]:
#обучаем модель на трейне с целевой метрикой MAPE, валидация для контроля переобучения на валидационном множестве
#выводить на просмотр каждую 100 итерацию алгоритма
#по умолчанию алгоритм пройдет 1000 итераций
model = CatBoostRegressor(cat_features=cat_features,
                          eval_metric='MAPE',
                          random_seed=42,
                          verbose=100)

model.fit(train[X],train[y],eval_set=(val[X],val[y]))

Learning rate set to 0.035473
0:	learn: 0.6672904	test: 0.9769368	best: 0.9769368 (0)	total: 25.3ms	remaining: 25.2s
100:	learn: 0.2443654	test: 0.3000888	best: 0.3000888 (100)	total: 2.67s	remaining: 23.7s
200:	learn: 0.1497616	test: 0.2299954	best: 0.2299954 (200)	total: 5.61s	remaining: 22.3s
300:	learn: 0.1008075	test: 0.2143059	best: 0.2143059 (300)	total: 8.57s	remaining: 19.9s
400:	learn: 0.0794897	test: 0.2054409	best: 0.2053971 (399)	total: 11.6s	remaining: 17.4s
500:	learn: 0.0626466	test: 0.2014763	best: 0.2014483 (496)	total: 14.7s	remaining: 14.6s
600:	learn: 0.0497040	test: 0.2005207	best: 0.2005054 (599)	total: 17.7s	remaining: 11.7s
700:	learn: 0.0387351	test: 0.2024124	best: 0.2005054 (599)	total: 20.8s	remaining: 8.86s
800:	learn: 0.0313423	test: 0.2015991	best: 0.2005054 (599)	total: 23.7s	remaining: 5.89s
900:	learn: 0.0261878	test: 0.2013861	best: 0.2005054 (599)	total: 26.9s	remaining: 2.95s
999:	learn: 0.0213115	test: 0.2004458	best: 0.2002628 (944)	total: 29.8s	

In [81]:
#точка останова итеративной модели
model.best_iteration_

944

In [82]:
#скорость обучения модели
model.learning_rate_

0.03547300025820732

In [83]:
#оцениваем качество прогноза на трейне
train['Продажи_прогноз'] = model.predict(train[X])

print('Ошибка прогноза на трейне:',mape(train['Продажи'],train['Продажи_прогноз']))

Ошибка прогноза на трейне: 0.1014385974929361


In [84]:
#прогнозируем обученной моделью на тесте
test['Продажи_прогноз'] = model.predict(test[X])

print('Ошибка прогноза на тесте:',mape(test['Продажи'],test['Продажи_прогноз']))

Ошибка прогноза на тесте: 0.239462057200235


In [85]:
#Переобучим модель на более полных данных - (трейн + вал), ди сих пор валидация использовалась для контроля переобучения модели
#Здесь валидации уже не будет
train_full = pd.concat([train,val])

#Парметры модели из результатов обучения
parameters = {'iterations': model.best_iteration_ + 1,
              'cat_features': cat_features,
              'eval_metric': 'MAPE',
              'learning_rate': model.learning_rate_,
              'random_seed':42,
              'verbose':100}

model = CatBoostRegressor(**parameters)

#обучение на полных данных
model.fit(train_full[X],train_full[y])

0:	learn: 0.6848640	total: 15.9ms	remaining: 15s
100:	learn: 0.2203494	total: 2.77s	remaining: 23.1s
200:	learn: 0.1348061	total: 5.62s	remaining: 20.8s
300:	learn: 0.0976613	total: 8.59s	remaining: 18.4s
400:	learn: 0.0754225	total: 11.5s	remaining: 15.6s
500:	learn: 0.0624195	total: 14.5s	remaining: 12.8s
600:	learn: 0.0507332	total: 17.4s	remaining: 9.95s
700:	learn: 0.0423827	total: 20.3s	remaining: 7.08s
800:	learn: 0.0358490	total: 23.3s	remaining: 4.18s
900:	learn: 0.0300814	total: 26.2s	remaining: 1.28s
944:	learn: 0.0284716	total: 27.4s	remaining: 0us


In [86]:
#оцениваем качество прогноза на трейне
train['Продажи_новый_прогноз'] = model.predict(train[X])

print('Ошибка нового прогноза на трейне:',mape(train['Продажи'],train['Продажи_новый_прогноз']))

Ошибка нового прогноза на трейне: 0.08519065846254362


In [87]:
#прогнозируем переобученной на полных данных моделью на тесте
test['Продажи_новый_прогноз'] = model.predict(test[X])
print('Ошибка нового прогноза на тесте:',mape(test['Продажи'],test['Продажи_новый_прогноз']))

Ошибка нового прогноза на тесте: 0.1817976379595769
